In [ ]:
# Copyright 2023 Google LLC
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# Set up in cloud

need to task a getsentry GH owner to make a fine-grained token for downloading this repo and we'll put it in a secret

In [3]:
import os

os.environ["GITHUB_TOKEN"] = ""

In [4]:
!pip install git+https://${GITHUB_TOKEN}@github.com/getsentry/grouping-trainer.git

  Cloning https://****@github.com/getsentry/grouping-trainer.git to /var/tmp/pip-req-build-bzamvggm
  Running command git clone --filter=blob:none --quiet 'https://****@github.com/getsentry/grouping-trainer.git' /var/tmp/pip-req-build-bzamvggm
  Resolved https://****@github.com/getsentry/grouping-trainer.git to commit f6976278b1cd6b0afa5b255222407d8e51f463fb
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 38.0/38.0 MB 136.7 MB/s  0:00:00m0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 132.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 167.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 888.0/888.0 MB 40.0 MB/s  0:00:09m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 594.3/594.3 MB 56.1 MB/s  0:00:05m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 19.9 MB/s  0:00:006m0:00:01


In [10]:
!gsutil -m cp -r gs://seer-models/models/issue_grouping_v1 .

Copying gs://seer-models/models/issue_grouping_v1/.DS_Store...
Copying gs://seer-models/models/issue_grouping_v1/data.pkl...                   
Copying gs://seer-models/models/issue_grouping_v1/embeddings/1_Pooling/config.json...
Copying gs://seer-models/models/issue_grouping_v1/embeddings/README.md...       
Copying gs://seer-models/models/issue_grouping_v1/embeddings/config.json...     
Copying gs://seer-models/models/issue_grouping_v1/embeddings/config_sentence_transformers.json...
Copying gs://seer-models/models/issue_grouping_v1/embeddings/modules.json...
Copying gs://seer-models/models/issue_grouping_v1/embeddings/configuration_bert.py...
Copying gs://seer-models/models/issue_grouping_v1/embeddings/merges.txt...      
Copying gs://seer-models/models/issue_grouping_v1/embeddings/model.safetensors...
Copying gs://seer-models/models/issue_grouping_v1/embeddings/sentence_bert_config.json...
Copying gs://seer-models/models/issue_grouping_v1/embeddings/special_tokens_map.json...
Copyin

In [11]:
!gsutil -m -o GSUtil:check_hashes=never cp -r gs://grouping-data/final_csvs .

Copying gs://grouping-data/final_csvs/synthetic-semi-easy-negatives.csv...
Copying gs://grouping-data/final_csvs/test.csv...                               
Copying gs://grouping-data/final_csvs/train.csv...                              
Copying gs://grouping-data/final_csvs/val.csv...                                
/ [4/5 files][  5.4 GiB/  5.4 GiB]  99% Done  99.9 MiB/s ETA 00:00:00           

# Set up

In [ ]:
# !rm -rf 2025-12-16-09-13-42-output

In [ ]:
%env WANDB_API_KEY=$(gcloud secrets versions access latest \
  --secret=wandb-api-key \
  --project=996102297610)

In [ ]:
import wandb

For some reason, you might need to run this next cell, interrupt it (it will hang), and then run it again (it will
immediately succeed)

In [4]:
wandb.login()

wandb: Currently logged in as: kush-dubey (sentry-seer) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [ ]:
import math
import warnings

from datetime import datetime
from sentence_transformers import SentenceTransformerTrainingArguments
from sentence_transformers.training_args import MultiDatasetBatchSamplers
import torch

import grouping_trainer as gt

In [6]:
assert torch.cuda.is_available()

In [7]:
timestamp = datetime.now().strftime("%Y-%m-%d-%H-%M-%S")

Some vars to care about

In [ ]:
SAMPLE_TRAIN: int | None = 100_000 if torch.cuda.is_available() else 30
SAMPLE_VAL: int | None = 2_000 if torch.cuda.is_available() else 20

OUTPUT_DIR = f"./{timestamp}-output"
PER_DEVICE_TRAIN_BATCH_SIZE = 256
GRADIENT_ACCUMULATION_STEPS = 1
GRADIENT_CHECKPOINTING = True  # disable for A100 80GB

PER_DEVICE_EVAL_BATCH_SIZE = 1
EVAL_STEPS = 75
PER_DEVICE_TOKEN_BUDGET = 8192  # try increasing for A100 80GB

In [9]:
assert (EVAL_STEPS % 5) == 0, "pls for sanity make it divisible by 5"

# Load model

In [10]:
gt.utils._cuda_empty_cache()

In [11]:
model_path = "issue_grouping_v1/embeddings"
# model_path = "/Users/kdubey/projects/seer/models/issue_grouping_v1/embeddings"
model = gt.utils.SentenceTransformer(str(model_path), trust_remote_code=True)
model.device

/opt/conda/lib/python3.10/site-packages/torch/onnx/_internal/registration.py:162: OnnxExporterWarning: Symbolic function 'aten::scaled_dot_product_attention' already registered for opset 14. Replacing the existing function with new function. This is unexpected. Please report it on https://github.com/pytorch/pytorch/issues.
  warnings.warn(


device(type='cuda', index=0)

In [12]:
assert "layernorm" in repr(model[0].auto_model).lower()
assert "batch" not in repr(model[0].auto_model).lower()

Don't have batch norm. That could mess up stuff for the deduplication strategy.

In [13]:
_ = model.encode("test")

# Load data

We'll make a `dataset_val` for val loss.

In [ ]:
dataset_val = gt.train.df_to_dataset(utils.load_val_df(sample_size=SAMPLE_VAL))
len(dataset_val)

In [ ]:
dataset_dict_train, frac_positive = utils.load_train_dataset_dict(
    sample_size=SAMPLE_TRAIN, min_dataset_size=PER_DEVICE_TRAIN_BATCH_SIZE
)
len(dataset_dict_train)

  0%|          | 0/145 [00:00<?, ?it/s]

# Set up `Trainer`

In [ ]:
evaluator = gt.evaluator.MinPrecisionEvaluator(
    sentences1=list(dataset_val["query_stacktrace_string"]),
    sentences2=list(dataset_val["candidate_stacktrace_string"]),
    labels=[int(record["label"]) for record in dataset_val],
    name="val",
    show_progress_bar=True,
    batch_size=PER_DEVICE_EVAL_BATCH_SIZE,
    truncate_dims=(64, 768),
)

Before training:

In [18]:
evaluator(model)

Batches:   0%|          | 0/3513 [00:00<?, ?it/s]

/opt/conda/lib/python3.10/site-packages/sentence_transformers/util/tensor.py:28: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /pytorch/torch/csrc/utils/tensor_new.cpp:253.)
  a = torch.tensor(a)


{'val_cosine_accuracy': 0.746,
 'val_cosine_accuracy_threshold': 0.9466277360916138,
 'val_cosine_f1': 0.8297604035308954,
 'val_cosine_f1_threshold': 0.9228835105895996,
 'val_cosine_precision': 0.7364297705651931,
 'val_cosine_recall': 0.9501805054151624,
 'val_cosine_ap': 0.8954993723969016,
 'val_cosine_mcc': 0.2757394853448247}

In [19]:
def init_bias(frac_positive: float):
    return math.log(frac_positive / (1 - frac_positive))

In [ ]:
trainer = gt.train.Trainer(
    model=model,
    args=SentenceTransformerTrainingArguments(
        # These should prolly be unchanged
        output_dir=OUTPUT_DIR,
        bf16=torch.cuda.is_bf16_supported(),
        fp16=False,
        dataloader_pin_memory=torch.cuda.is_available(),
        num_train_epochs=1,
        # Save memory
        gradient_checkpointing=GRADIENT_CHECKPOINTING,
        gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
        #
        # Datalaoder
        multi_dataset_batch_sampler=MultiDatasetBatchSamplers.PROPORTIONAL,
        # Each iter, pick a project randomly, sample from it.
        # Next iter, pick another project randomly, sample from it, etc.
        per_device_train_batch_size=PER_DEVICE_TRAIN_BATCH_SIZE,
        seed=42,  # passed to batch sampler
        #
        # Optimizer
        learning_rate=2e-5,
        learning_rate_mapping={
            # These are important to tune. Higher so that training doesn't get stuck. TODO: check
            r"^log_scale$": 2e-4,
            r"^bias$": 2e-4,
        },
        weight_decay=0.01,
        warmup_ratio=0.1,
        #
        # Eval
        per_device_eval_batch_size=PER_DEVICE_EVAL_BATCH_SIZE,
        eval_strategy="steps",
        eval_steps=EVAL_STEPS,
        #
        # Logging
        logging_strategy="steps",
        logging_steps=EVAL_STEPS // 10,  # train loss alongside metrics table
        run_name=f"{timestamp}-grouping-trainer",
        report_to="wandb",
        #
        # Checkpointing
        save_strategy="steps",
        save_steps=EVAL_STEPS // 5,
        save_total_limit=2,
    ),
    #
    # Training
    loss=gt.train.SigmoidPairwiseLoss(
        model,
        bias_init=init_bias(frac_positive),
        matryoshka_dims=[768, 512, 256, 128, 64],
        matryoshka_weights=[2, 1, 1, 0.5, 0.25],
        n_dims_per_step=2,
    ),
    data_collator=gt.train.DefaulDataCollator(tokenize_fn=model.tokenize),
    train_dataset=dataset_dict_train,
    shuffle_within_dataset=False,  # more cache hits in each forward
    per_device_token_budget=PER_DEVICE_TOKEN_BUDGET,
    #
    # Evaluator
    eval_dataset=dataset_val,  # val loss
    evaluator=evaluator,  # val recall at x precision
)

In [ ]:
warnings.filterwarnings(
    "ignore",
    message=".*torch.utils.checkpoint: the use_reentrant parameter.*",
    category=UserWarning,
)

In [22]:
train_output = trainer.train()

You are using an old version of the checkpointing format that is deprecated (We will also silently ignore `gradient_checkpointing_kwargs` in case you passed it).Please update to the new format on your modeling file. To use the new format, you need to completely remove the definition of the method `_set_gradient_checkpointing` in your model.


Step,Training Loss,Validation Loss
75,0.580000,0.705410
150,0.368600,0.630100
225,0.406900,0.567888
300,0.375300,0.579943
375,0.327300,0.562493
450,0.326000,0.549743


In [ ]:
!gsutil -m rsync -r {OUTPUT_DIR} gs://grouping-data/runs/{OUTPUT_DIR}/training

CommandException: The rsync command requires at least 2 arguments. Usage:

  gsutil rsync [OPTION]... src_url dst_url

For additional help run:
  gsutil help rsync


In [ ]:
!gsutil -m cp -r notebook_template.ipynb gs://grouping-data/runs/{OUTPUT_DIR}

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Copying file://notebook_template.ipynb [Content-Type=application/octet-stream]...
/ [1/1 files][ 58.6 KiB/ 58.6 KiB] 100% Done                                    
Operation completed over 1 objects/58.6 KiB.                                     
